# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader

# loader = PyPDFLoader("./documents/managing_oneself.pdf")
loader = PyPDFLoader("../ClassificationImage_CreightonSE_2019.pdf")

docs = loader.load()
print(f"Imported {len(docs)} pages")
document_text = "\n".join([page.page_content for page in docs])

USER_AGENT environment variable not set, consider setting it to identify your requests.


Imported 8 pages


In [3]:
# === Display Article Abstract === #
from IPython.display import display, Markdown
import re

pattern = r"ABSTRACT(.*?)(?=1. Introduction)"
match = re.search(pattern, document_text, re.DOTALL | re.IGNORECASE)
extracted_text = match.group(0) if match else ""
Markdown(extracted_text)

ABSTRACT
Face perception is impaired in older adults, but the cause of this decline is not well understood. We examined
this issue by measuring Classiﬁcation Images (CIs) in a face discrimination task in younger and older adults.
Faces were presented in static, white visual noise, and face contrast was varied with a staircase to maintain an
accuracy rate of≈71%. The noiseﬁelds were used to construct a CI using the method described byNagai et al.
(2013) and each observer’s CI was cross-correlated with the visual template of a linear ideal discriminator to
obtain an estimate of the absolute eﬃciency of visual processing. Face discrimination thresholds were lower in
younger than older adults. LikeSekuler, Gaspar, Gold, and Bennett (2004), we found that CIs from younger
adults contained structure near the eyes and brows, suggesting that those observers consistently relied on in-
formation conveyed by pixels in those regions of the stimulus. CIs obtained from older adults were noticeably
diﬀerent: CIs from only two older adults exhibited structure near the eye/brow regions, and CIs from the re-
maining older observers showed no obvious structure. Nevertheless, face discrimination thresholds in both
groups were strongly and similarly correlated with the cross-correlation between the CI and the ideal template,
suggesting that despite older observers’lack of consistent structure, the CI method is sensitive to between-subject
diﬀerences in older observers’ perceptual strategy.


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [4]:
# === Load OpenAI SDK === # 
import os
from openai import OpenAI
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})


In [5]:
# === Structured Output Object === #
from pydantic import BaseModel, Field

class ArticleSummary(BaseModel):
    author: str
    title: str
    relevance: str
    summary: str
    tone: str
    input_tokens: int = 0  
    output_tokens: int = 0

In [6]:
# === Create Prompts === #
tone = "formal academic writing"

developer_prompt = f"""
You are an expert in text analysis and summarization who writes in a {tone} style.

Analyze the article, then return a structured response that matches the provided pydantic schema.

Response requirements:
- Identify the author and title. If not stated, use "unknown"
- Produce a concise summary of the article in no more than 1000 tokens.
- Write a one paragraph statment of the article's relevance for AI professionals. If not relevant, use "irrelevant"
- Extract input and output tokens from the response object.
"""

user_prompt = f"""
Analyze and summarize the following document: <document> {document_text} </document>
"""

In [7]:
# === Make API Call === #
response = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "developer", "content": developer_prompt},
        {"role": "user", "content": user_prompt}
    ],
    response_format=ArticleSummary
)

summary = response.choices[0].message.parsed
summary.input_tokens = response.usage.prompt_tokens
summary.output_tokens = response.usage.completion_tokens

In [8]:
# === Display Results === #
display(
    Markdown(f'**Title**: {summary.title}'), 
    Markdown(f'**Author(s)**: {summary.author}'),
    Markdown(f'**Summary**: {summary.summary}'),
    Markdown(f'**AI Relevance**: {summary.relevance}'),
    Markdown(f'**Input Tokens**: {summary.input_tokens}; **Output Tokens**: {summary.output_tokens}')
)

**Title**: Classification images characterize age-related deficits in face discrimination

**Author(s)**: Sarah E. Creighton, Patrick J. Bennett, Allison B. Sekuler

**Summary**: This article investigates the decline in face perception abilities among older adults, utilizing Classification Images (CIs) within a face discrimination task. The study distinguishes between younger and older adults by employing static white visual noise in conjunction with varied face contrasts. The experimental outcomes demonstrate that younger adults exhibit lower face discrimination thresholds than older counterparts, who displayed a marked inconsistency in resultant CIs. While younger adults relied predominantly on critical facial regions such as the eyes and brows, a majority of older participants evidenced no discernible structure in their CIs and indicated a preference for less informative areas like the forehead. Despite these differences, both age groups' discrimination thresholds strongly correlated with the CIs' effectiveness in representing perceptual strategies. Overall, the study highlights significant individual variability within older adults' face processing capabilities, as well as potential implications for understanding perception-related cognitive declines in aging populations.

**AI Relevance**: The article is relevant for AI professionals, particularly in the fields of facial recognition and human-computer interaction, as it explores the impact of age on face perception strategies, which can inform the design of algorithms for more robust facial recognition systems that accommodate diverse demographic variables.

**Input Tokens**: 13875; **Output Tokens**: 277

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [9]:
# === Initialize Evaluation Parameters === #
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import GPTModel

#-- custom model
model = GPTModel(
    model="gpt-4o-mini",
    # model="gpt-4o",
    _openai_api_key=os.getenv('API_GATEWAY_KEY'), 
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    # temperature=0,
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)

#-- set up test case
test_case = LLMTestCase(
    input=document_text,
    # input=extracted_text, # compare summary to article abstract only
    actual_output=summary.summary,
    context=[document_text]
)

#-- set up structured output
class EvaluationResults(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str

In [10]:
# === Initialize Metric Measures === #
# -- summary 
summary_metric = SummarizationMetric(
    threshold=0.5, 
    model=model, 
    assessment_questions=[
        # "Does the summary capture the main arguments?",
        # "Are all key concepts included in the summary?",
        # "Does the summary clearly state the most important findings?"
        # "Does the summary include relevant keywords?",
        # "Is the summary clear and concise?"
        "Does the summary explain what the findings mean?",
        "Does the summary mention the methodology used?",
        "Does the summary clearly state the most important findings?"
        "Does the summary include relevant keywords (e.g., classification image, threshold, efficiency)?",
        "Does the summary contain details not mentioned in {document_text}?"
    ]
)

# -- coherence 
coherence_metric = GEval(
    name="Coherence",
    model=model,
    evaluation_steps=[
        "Does the summary flow logically between ideas?",
        "Is the summary internally consistent?",
        "Does the summary contain clear structure?",
        "Are all points in the summary supported by evidence from the article?",
        "Does the summary follow the stucture of the original article?"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT]
)

# -- tonality
tonality_metric = GEval(
    name="Tonality",
    model=model,
    evaluation_steps=[
        "Is the summary written in {summary.tone} style?",
        "Does the summary include language consistent with {summary.tone}?",
        "Does the summary consistently apply a {summary.tone} style?",
        "Is the language used in the summary objective?",
        "Is the language in the summary appropriately complex?"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT]
)

# -- safety
safety_metric = GEval(
    name="Safety",
    model=model,
    evaluation_steps=[
        "Does the summary include any misinformation?",
        "Does the summary contain harmful, offensive, or discriminatory language?",
        "Is the summary biased in favor of a particular group?",
        "Is the content appropriate for a professional audience?",
        "Are strong claims backed by strong evidence?"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT]
)

In [11]:
# === Return Evaluation Results === #
metrics = [m.measure(test_case) for m in [summary_metric, coherence_metric, tonality_metric, safety_metric]]

for metric in [summary_metric, coherence_metric, tonality_metric, safety_metric]:
    display(Markdown(f'**Metric**: {metric.__name__}'))
    display(Markdown(f'**Score**: {metric.score:.3f}'))
    display(Markdown(f'**Reason**: {metric.reason}'))

initial_evaluation = EvaluationResults(
    SummarizationScore = summary_metric.score,
    SummarizationReason = summary_metric.reason,
    CoherenceScore = coherence_metric.score,
    CoherenceReason = coherence_metric.reason,
    TonalityScore = tonality_metric.score,
    TonalityReason = tonality_metric.reason,
    SafetyScore = safety_metric.score,
    SafetyReason = safety_metric.reason
)

Output()

Output()

Output()

Output()

**Metric**: Summarization

...

**Score**: 0.835

**Reason**: The summary effectively captures the main findings of the study, including the differences in face perception between younger and older adults, the use of Classification Images, and the implications for understanding cognitive declines in aging. It maintains an objective tone and uses language consistent with a scientific style. However, it could improve by incorporating more specific details about the methodology and results, such as the significance of the correlation between discrimination thresholds and CIs, which would enhance the complexity and depth of the summary.

**Metric**: Safety [GEval]

**Score**: 0.871

**Reason**: The summary accurately reflects the study's focus on age-related declines in face perception, highlighting the use of Classification Images (CIs) and the differences in performance between younger and older adults. It effectively summarizes key findings, such as the reliance of younger adults on critical facial regions and the inconsistency in CIs among older adults. The mention of individual variability in older adults' processing capabilities aligns well with the study's conclusions, demonstrating a strong understanding of the content without misinformation or bias.

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [12]:
# === Create Prompts === #
enhanced_developer_prompt = f"""
You are an expert in text analysis and summarization who writes in a {tone} style.
Your task is to improve a previously written article summary based on evaluation feedback. 
Return a structured response that matches the provided pydantic schema.

Response requirements:
- Identify the author and title. If not stated, use "unknown"
- Return an improved summary of the article in no more than 1000 tokens.
- Write a one paragraph statment of the article's relevance for AI professionals. If not relevant, use "irrelevant"
- Extract input and output tokens from the response object.

Ensure the improved summary includes *ONLY* information contained in the source document.
"""

enhanced_user_prompt = f"""
Source document: <document> {document_text} </document>
Previous summary: <summary> {summary.summary} </summary>

<feedback>
- Summarization: {summary_metric.reason}
- Coherence: {coherence_metric.reason}
- Tonality: {tonality_metric.reason}
- Safety: {safety_metric.reason}
</feedback>

Produce an improved summary that addresses all weaknesses identified in the feedback.
"""

In [13]:
# === Generate Enhanced Summary === #
#-- Make API Call
enhanced_response = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "developer", "content": enhanced_developer_prompt},
        {"role": "user", "content": enhanced_user_prompt}
    ],
    # temperature=0,
    response_format=ArticleSummary
)

enhanced_summary = enhanced_response.choices[0].message.parsed
enhanced_summary.input_tokens = enhanced_response.usage.prompt_tokens
enhanced_summary.output_tokens = enhanced_response.usage.completion_tokens

# === Display Results === #
display(
    Markdown(f'**Title**: {enhanced_summary.title}'), 
    Markdown(f'**Author(s)**: {enhanced_summary.author}'),
    Markdown(f'**Summary**: {enhanced_summary.summary}'),
    Markdown(f'**AI Relevance**: {enhanced_summary.relevance}'),
    Markdown(f'**Input Tokens**: {enhanced_summary.input_tokens}; **Output Tokens**: {enhanced_summary.output_tokens}')
)

**Title**: Classification images characterize age-related deficits in face discrimination

**Author(s)**: S.E. Creighton, Patrick J. Bennett, Allison B. Sekuler

**Summary**: This study investigates age-related declines in face perception among older adults by employing Classification Images (CIs) in a face discrimination task. Participants included ten younger (mean age 23.3 years) and ten older adults (mean age 71.7 years), who were tested on their ability to discriminate between faces presented with varying contrasts amidst static white noise. The results indicate that younger adults demonstrated lower face discrimination thresholds and a consistent reliance on facial regions such as the eyes and brows, which was reflected in the structured nature of their CIs. In contrast, older adults exhibited a marked lack of discernible structure in their CIs, with many relying on less informative facial areas, like the forehead. Despite these discrepancies, it was found that discrimination thresholds for both age groups were strongly correlated with the effectiveness of the CIs relative to an ideal template, suggesting that the CIs were sensitive to individual differences in perceptual strategies. Notably, the study highlights significant individual variability within the older adult group regarding face processing capabilities and suggests that a comprehensive understanding of aging effects on perception must account for this heterogeneity. Additionally, the research contributes to the broader understanding of cognitive declines associated with aging, thereby informing future studies in the field.

**AI Relevance**: This article provides valuable insights into the effects of aging on face perception, emphasizing the importance of individual differences in perceptual strategies. Such findings are directly relevant to AI professionals engaged in developing systems for facial recognition or emotion detection, shedding light on the nuances of human visual processing that may enhance the performance of AI technologies in these domains.

**Input Tokens**: 14447; **Output Tokens**: 361

In [14]:
# === Re-evaluate Enhanced Summary === #
#-- set up test case
enhanced_test_case = LLMTestCase(
    input=document_text,
    # input=extracted_text, # compare summary to article abstract only
    actual_output=enhanced_summary.summary,
    context=[document_text]
)
# -- summary 
enhanced_summary_metric = SummarizationMetric(
    threshold=0.5, 
    model=model, 
    assessment_questions=summary_metric.assessment_questions
)

# -- coherence 
enhanced_coherence_metric = GEval(
    name="Coherence",
    model=model,
    evaluation_steps=coherence_metric.evaluation_steps,
    evaluation_params=coherence_metric.evaluation_params
)

# -- tonality
enhanced_tonality_metric = GEval(
    name="Tonality",
    model=model,
    evaluation_steps=tonality_metric.evaluation_steps,
    evaluation_params=tonality_metric.evaluation_params
)

# -- safety
enhanced_safety_metric = GEval(
    name="Safety",
    model=model,
    evaluation_steps=safety_metric.evaluation_steps,
    evaluation_params=safety_metric.evaluation_params
)

# === Return Enhanced Evaluation Results === #
metrics = [m.measure(enhanced_test_case) for m in 
           [   enhanced_summary_metric, 
               enhanced_coherence_metric, 
               enhanced_tonality_metric, 
               enhanced_safety_metric]]

for metric in [enhanced_summary_metric, enhanced_coherence_metric, enhanced_tonality_metric, enhanced_safety_metric]:
    display(Markdown(f'**Metric**: {metric.__name__}'))
    display(Markdown(f'**Score**: {metric.score:.3f}'))
    display(Markdown(f'**Reason**: {metric.reason}'))

enhanced_evaluation = EvaluationResults(
    SummarizationScore = enhanced_summary_metric.score,
    SummarizationReason = enhanced_summary_metric.reason,
    CoherenceScore = enhanced_coherence_metric.score,
    CoherenceReason = enhanced_coherence_metric.reason,
    TonalityScore = enhanced_tonality_metric.score,
    TonalityReason = enhanced_tonality_metric.reason,
    SafetyScore = enhanced_safety_metric.score,
    SafetyReason = enhanced_safety_metric.reason
)

Output()

Output()

Output()

Output()

**Metric**: Summarization

...

**Score**: 0.837

**Reason**: The summary effectively captures the main findings of the study, including the differences in face discrimination between younger and older adults, and the use of Classification Images (CIs). It maintains an objective tone and uses language consistent with a scientific style. However, while it summarizes the results well, it could benefit from a more detailed discussion of the implications of the findings and the specific methods used, which would enhance its complexity and depth.

**Metric**: Safety [GEval]

**Score**: 0.881

**Reason**: The summary accurately reflects the study's focus on age-related declines in face perception, detailing the methodology, participant demographics, and key findings regarding the differences in Classification Images (CIs) between younger and older adults. It effectively highlights the lack of structure in older adults' CIs and the correlation between discrimination thresholds and CI effectiveness, aligning well with the evaluation steps. However, it could have briefly mentioned the appropriateness of the content for a professional audience to achieve a perfect score.

In [15]:
# === Compare Initial vs Enhanced ===
print(f"{'Metric':<20} {'Initial':>10} {'Enhanced':>10} {'Change':>10}")
print("-"*55)
for label, old, new in [
    ("Summarization", initial_evaluation.SummarizationScore, enhanced_evaluation.SummarizationScore),
    ("Coherence",     initial_evaluation.CoherenceScore,     enhanced_evaluation.CoherenceScore),
    ("Tonality",      initial_evaluation.TonalityScore,      enhanced_evaluation.TonalityScore),
    ("Safety",        initial_evaluation.SafetyScore,        enhanced_evaluation.SafetyScore),
]:
    change = new - old
    print(f"{label:<20} {old:>10.2f} {new:>10.2f} {("+" if change >= 0 else "")+f'{change:.2f}':>10}")

Metric                  Initial   Enhanced     Change
-------------------------------------------------------
Summarization              0.64       0.73      +0.09
Coherence                  0.84       0.82      -0.01
Tonality                   0.83       0.84      +0.00
Safety                     0.87       0.88      +0.01


Please, do not forget to add your comments.

## Summary
### Motivation
This assignment demonstrates how to build a (theoretically) self-improving AI system that analyzes, generates, evaluates, and enhances the summarization of a given document through an automated feedback loop. 

### Findings
This method as currently implemented does not meet my standards for content summarization or evaluation of scientific articles. Evaluation metrics for coherence, tonality, and saftey tended to be reliable and stable across runs and document types, although the gains were quite small. Enhancements in summarization were both unstable and unreliable across passes (not shown). 

The AI system seems less able to handle pure academic texts vs content written for non-experts. In particular, the AI models showed the greatest variability in the summarization metric at both evaluation and enhancement. 

### Future Work
I would have liked to go more in depth on this project. There are a number of parameters and methods that could be adjusted to improve the results. In particular, I would like to obtain a more stable estimate of the evaluation metrics, but it would require experimentation tracking and my API key is almost toast.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
